#Processign Silver layer for account_move_line model

For create the silver account_move_line I get temp view for load all data from the bronze layer, and complete the 

In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.account_move_line

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_account_move_line AS
SELECT 
id AS move_line_id,
move_id[0] AS move_id,
CAST(date AS DATE) AS date,
account_id[0] AS account_id,
journal_id[0] AS journal_id,
CAST(CASE
        WHEN partner_id IS NULL OR partner_id = 'false' THEN NULL
        ELSE get_json_object(partner_id, '$[0]')
        END AS INT
    ) AS partner_id,
amount_currency,
amount_residual,
balance,
credit,
debit,
ref,
CAST(create_date AS TIMESTAMP) AS created_at,
CAST(write_date AS TIMESTAMP) AS updated_at,
processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_move_line
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id 
    ORDER BY write_date DESC, processing_timestamp DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_move_line (
    move_line_id BIGINT,
    date DATE,
    move_id BIGINT,
    account_id INT,
    journal_id INT,
    partner_id BIGINT,
    amount_currency DECIMAL(10,2),
    amount_residual DECIMAL(10, 2),
    balance FLOAT,
    credit FLOAT,
    debit FLOAT,
    ref STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move_line AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_account_move_line
) AS source
ON target.move_line_id = source.move_line_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    move_id = source.move_id,
    date = source.date,
    account_id = source.account_id,
    journal_id = source.journal_id,
    partner_id = source.partner_id,
    amount_currency = source.amount_currency,
    amount_residual = source.amount_residual,
    balance = source.balance,
    credit = source.credit,
    debit = source.debit,
    ref = source.ref,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN 
  INSERT (
    move_line_id,
    move_id,
    date,
    account_id,
    journal_id,
    partner_id,
    amount_currency,
    amount_residual,
    balance,
    credit,
    debit,
    ref,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.move_line_id,
    source.move_id,
    source.date,
    source.account_id,
    source.journal_id,
    source.partner_id,
    source.amount_currency,
    source.amount_residual,
    source.balance,
    source.credit,
    source.debit,
    source.ref,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );


In [0]:
%sql
DROP VIEW IF EXISTS stg_delete_account_move

In [0]:
%sql

CREATE TEMP VIEW stg_delete_account_move AS
SELECT * FROM bakery_pos_develop.silver.account_move WHERE state = 'deleted'


In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move_line AS target
USING stg_delete_account_move AS source
ON target.move_id = source.account_move_id
WHEN MATCHED THEN 
  UPDATE SET
    updated_by = source.updated_by,
    state = 'deleted',
    silver_processed_at = source.silver_processed_at;
